# 第 3 课：MCP Server —— 把工具变成“插件”

第 2 课的工具是写死在 notebook 里的：`TOOLS` 说明书 + `TOOL_IMPL` 方法表，只有我们自己那个 Agent 能用。

这一课把它们搬进一个**独立运行的 MCP Server**（[03_order_server.py](03_order_server.py)），然后：
1. 用 MCP 客户端连上它，看看协议里到底传了什么
2. 把第 2 课的 Agent 循环改成“工具从 MCP 来”
3. 把同一个 Server 接进 Claude Code，让**别人的 Agent** 用你的工具

**这一课学到的**
- MCP：Server / Client / Host 三个角色、tools 和 resources、stdio 传输、工具报错的正确姿势
- Python：`async` / `await`、`async with`、装饰器 `@`、`Annotated` 类型标注、`if __name__ == "__main__"`

| C# 里你熟悉的 | 这一课的 Python 写法 |
|---|---|
| NuGet `ModelContextProtocol`，`AddMcpServer().WithStdioServerTransport()` | `pip`/`uv` 包 `mcp`，`MCPServer(...)` + `mcp.run()` |
| 方法上标 `[McpServerTool]` | 函数上加装饰器 `@mcp.tool()` |
| 参数上标 `[Description("...")]` | `Annotated[str, Field(description="...")]` |
| `async Task` / `await` | `async def` / `await`（几乎一模一样） |
| `await using var client = ...` | `async with Client(...) as client:` |
| `static void Main()` | `if __name__ == "__main__":` |

## 1. 先看图：MCP 解决什么问题

没有 MCP 时，每个 Agent 都要自己接一遍每个系统（N 个 Agent × M 个系统）。MCP 把“工具”变成一个**标准协议**，和 USB 一样：系统只需要做一次 Server，所有支持 MCP 的 Agent 都能插上用。

```
 第 2 课：工具长在 Agent 里                    第 3 课：工具在独立进程里，用 MCP 协议通信

 ┌──────────── notebook ───────────┐         ┌── Host（Claude Code / 你的 notebook）──┐
 │ Agent 循环                       │         │  LLM  ⇄  Agent 循环                    │
 │   ├─ TOOLS（手写 JSON Schema）   │         │            │                          │
 │   └─ TOOL_IMPL（函数字典）       │         │        MCP Client                      │
 │         get_order()             │         └────────────┼──────────────────────────┘
 │         get_shipment()          │                      │ JSON-RPC（stdio 管道 或 HTTP）
 │         find_orders_by_customer()│         ┌────────────┴──────────────────────────┐
 └─────────────────────────────────┘         │ MCP Server（03_order_server.py）        │
                                             │   tools:     get_order / get_shipment  │
                                             │              find_orders_by_customer   │
                                             │   resources: orders://summary          │
                                             └───────────────────────────────────────┘
```

**三个角色**
- **Host**：用户在用的应用，里面有 LLM 和 Agent 循环（Claude Code、Claude Desktop、我们这个 notebook）
- **Client**：Host 里负责和某一个 Server 通信的连接对象，一个 Server 对应一个 Client
- **Server**：暴露能力的独立进程。它**不调用 LLM**，也不知道是谁在用它，只是老老实实地“列出工具、执行工具”

**一次调用的全过程**（注意模型依然从不直接碰数据，这一点和第 2 课完全一样）

```
 Host                 MCP Client                MCP Server
  │  启动子进程 ───────────────────────────────▶ │  uv run 03_order_server.py
  │                      │ ── initialize ──────▶ │  握手：协议版本、我有 tools/resources
  │                      │ ── tools/list ──────▶ │
  │  ◀─ 工具清单 ─────── │ ◀─ [get_order, ...] ─ │  JSON Schema 由 SDK 从函数签名自动生成
  │  把清单发给 LLM，LLM 说：调 get_order("1024")
  │                      │ ── tools/call ──────▶ │  执行你的 Python 函数
  │  ◀─ 结果 ─────────── │ ◀─ {order...} ─────── │
  │  把结果放回对话，继续 Agent 循环（和第 2 课一样）
```

MCP Server 能暴露三种东西，这一课主要用 tools：

| 原语 | 谁决定用 | 类比 |
|---|---|---|
| **tools** 工具 | 模型自己决定调用 | 应用服务的方法（可能有副作用） |
| **resources** 资源 | 用户/应用挑选后放进上下文 | 只读的文件或 GET 接口 |
| **prompts** 提示模板 | 用户主动选（如 `/` 命令） | 预置的操作手册 |

## 2. 读一下 Server 的代码

先打开 [03_order_server.py](03_order_server.py) 看一遍，不到 80 行。和第 2 课对比，变化只有三处：

1. 函数上多了 `@mcp.tool()` —— **不用再手写 `TOOLS` 那份 JSON Schema 了**，SDK 从函数名、参数类型、`Field(description=...)` 和 docstring 自动生成
2. 找不到订单时 `raise ToolError(...)` —— 这句话会原样交给模型，让它自己纠正
3. 最后一行 `mcp.run()` —— 默认用 **stdio**：Host 把 Server 当子进程启动，通过 stdin/stdout 收发 JSON。所以 Server 里**不能 `print()` 到 stdout**，日志要写 stderr

> 它是 `.py` 文件而不是 notebook，因为 Server 要作为一个独立进程长期运行，由 Host 启动。

## 3. 连上 Server，看看工具清单

`Client(StdioServerParameters(...))` 会把 Server 作为子进程启动并完成握手。`async with` 结束时自动关掉子进程，相当于 C# 的 `await using`。

> Jupyter 里可以直接在格子顶层写 `await`，普通 `.py` 文件里要用 `asyncio.run(main())` 包起来。

In [ ]:
import json
import sys
from mcp import Client, StdioServerParameters

# 怎么启动 Server：用当前这个 Python（.venv 里的）运行 03_order_server.py
SERVER = StdioServerParameters(command=sys.executable, args=["03_order_server.py"])

async with Client(SERVER) as mcp_client:
    print("连上了：", mcp_client.server_info.name, "| 协议版本", mcp_client.protocol_version)
    print("服务说明：", mcp_client.instructions)
    tools = (await mcp_client.list_tools()).tools

for t in tools:
    print(f"\n🔧 {t.name}：{t.description}")
    print("   参数 Schema：", json.dumps(t.input_schema, ensure_ascii=False))

对比第 2 课手写的 `tool("get_order", "...", {"order_id": "..."})`：这份 Schema 是 SDK 从 Python 类型标注自动生成的，和 ASP.NET Core 从 C# 方法签名生成 Swagger 是一个思路。

## 4. 不经过 LLM，直接调工具

MCP Server 本身和 LLM 无关。先手动调几次，包括一次故意出错的，看 `is_error` 和错误信息。

In [ ]:
def result_text(result) -> str:
    """MCP 的结果是一组 content 块（文字、图片……），这里把文字块拼起来。"""
    return "\n".join(c.text for c in result.content if c.type == "text")


async with Client(SERVER) as mcp_client:
    for name, args in [
        ("get_order", {"order_id": "1024"}),
        ("find_orders_by_customer", {"customer_name": "星辰"}),
        ("get_order", {"order_id": "9999"}),   # 故意查一个不存在的
    ]:
        r = await mcp_client.call_tool(name, args)
        print(f"{'❌' if r.is_error else '✅'} {name}({args})\n{result_text(r)}\n")

    # 顺便看一下 resource：它不是给模型“调用”的，而是让应用读出来放进上下文
    res = await mcp_client.read_resource("orders://summary")
    print("📄 orders://summary\n" + res.contents[0].text)

## 5. 核心：让第 2 课的 Agent 循环用 MCP 工具

只需要两处改动：

| 第 2 课 | 第 3 课 |
|---|---|
| `TOOLS` 手写的工具说明书 | `tools/list` 拿到的清单，**转换一下格式**交给 LLM |
| `TOOL_IMPL[name](**args)` 本地调函数 | `await mcp_client.call_tool(name, args)` 远程调 |

MCP 的工具格式和 OpenAI 的 function calling 格式几乎一样，转换只是换个“信封”：

In [ ]:
def to_openai_tool(t) -> dict:
    """MCP 工具定义 → OpenAI tools 参数里的一项。input_schema 本身就是 JSON Schema，原样放进去。"""
    return {
        "type": "function",
        "function": {"name": t.name, "description": t.description or "", "parameters": t.input_schema},
    }

print(json.dumps(to_openai_tool(tools[0]), ensure_ascii=False, indent=2))

连接大模型（和前两课一样）：

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

llm = OpenAI(
    api_key=os.environ["LLM_API_KEY"],
    base_url=os.getenv("LLM_BASE_URL", "https://open.bigmodel.cn/api/paas/v4/"),
)
MODEL = os.getenv("LLM_MODEL", "glm-4-flash")
print("使用模型:", MODEL)

下面这个 `run_agent` 和第 2 课的结构完全一样，变化的行都标了 `# ← MCP`。

函数前面多了 `async`，因为和 Server 通信是异步的。规则和 C# 一样：调用 `async` 函数要 `await`。

In [ ]:
SYSTEM = "你是订单客服助手。回答前必须用工具查询真实数据，不要编造。回答简洁，说明原因和下一步建议。"


async def run_agent(question: str, max_steps: int = 5) -> str:
    async with Client(SERVER) as mcp_client:                                          # ← MCP：连上 Server
        tools = [to_openai_tool(t) for t in (await mcp_client.list_tools()).tools]    # ← MCP：工具清单从 Server 来

        messages = [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": question},
        ]

        for step in range(1, max_steps + 1):
            resp = llm.chat.completions.create(model=MODEL, messages=messages, tools=tools, temperature=0)
            msg = resp.choices[0].message

            assistant = {"role": "assistant", "content": msg.content or ""}
            if msg.tool_calls:
                assistant["tool_calls"] = [
                    {"id": c.id, "type": "function", "function": {"name": c.function.name, "arguments": c.function.arguments}}
                    for c in msg.tool_calls
                ]
            messages.append(assistant)

            if not msg.tool_calls:
                return msg.content

            for call in msg.tool_calls:
                name = call.function.name
                args = json.loads(call.function.arguments or "{}")
                print(f"  🔧 第 {step} 步：调用 {name}({args})")

                result = await mcp_client.call_tool(name, args)                       # ← MCP：远程执行
                text = result_text(result)
                print(f"     ↳ {'❌ ' if result.is_error else ''}{' '.join(text.split())[:100]}")

                messages.append({"role": "tool", "tool_call_id": call.id, "content": text})

        return "（达到最大步数，已停止）"

In [ ]:
print("\n🤖", await run_agent("订单 1024 为什么还没发货？"))

In [ ]:
print("\n🤖", await run_agent("星辰软件一共有几个订单，各是什么状态？"))

注意 Agent 这边**一行工具代码都没有了**。以后 Server 加一个工具，Agent 不用改任何代码，下次 `list_tools` 就能看到。这就是 MCP 的价值：**工具和 Agent 解耦**。

## 6. 让别人的 Agent 用你的工具

同一个 `03_order_server.py`，不改一行，就能接进任何支持 MCP 的 Host。下面的命令在**终端**里运行（不是 notebook），路径换成你本机仓库的绝对路径。

**① MCP Inspector：官方调试工具（相当于 MCP 版的 Swagger UI）**

```bash
npx @modelcontextprotocol/inspector uv run 03_order_server.py
```
浏览器打开它给的地址，可以点着看工具清单、手动调用、看原始 JSON-RPC 消息。

**② Claude Code**

```bash
claude mcp add order-service -- uv run --directory /你的路径/agent-learning-py 03_order_server.py
claude mcp list          # 看到 order-service ✓ Connected 就成功了
```
然后在 Claude Code 里直接问：“订单 1024 为什么还没发货？”，它会请求你允许调用 `get_order`。

**③ Claude Desktop / Cursor 等**：在它们的 MCP 配置 JSON 里加一段，格式基本都是：

```json
{
  "mcpServers": {
    "order-service": {
      "command": "uv",
      "args": ["run", "--directory", "/你的路径/agent-learning-py", "03_order_server.py"]
    }
  }
}
```

> 这三种 Host 和我们的 notebook 做的事情一模一样：启动子进程 → `tools/list` → 把清单交给自己的模型 → 模型要调时发 `tools/call`。

## 7. 几条经验

- **Server 不调 LLM**：它只是工具的提供者。所以 MCP Server 是普通的后端开发，可以照常写单元测试。
- **工具描述就是给模型看的 API 文档**：描述写清楚“什么时候用、参数长什么样、返回什么”。
- **错误信息要对模型友好**：`ToolError("找不到订单 9999，请确认订单号，或先用 find_orders_by_customer 查询")` 比 `"404"` 有用得多，模型会照着提示自己纠正。
- **先只读，写操作要谨慎**：写操作（取消订单、退款）放进 MCP 后，任何接入的 Agent 都能调。Host 一般会让用户确认，但 Server 里自己也要做权限检查。
- **stdio 还是 HTTP**：本机给自己用选 stdio（最简单，不用管端口和鉴权）；给团队共享、部署到服务器选 Streamable HTTP（`mcp.run("streamable-http")`），这时就需要鉴权了。

## 动手练习

1. 在 `03_order_server.py` 里加一个工具 `get_customer_balance(customer_name)`，**不改 notebook 的任何代码**，重新运行第 5 节问“星辰软件还欠多少钱？”。
2. 把 `get_order` 里的 `ToolError` 换成 `ValueError`，再跑第 4 节，看模型收到的错误信息有什么不同，想想为什么 SDK 要这样设计。
3. 加一个写操作工具 `cancel_order(order_id, reason)`，接进 Claude Code，观察它在调用前怎么向你确认。
4. 用 MCP Inspector 打开 Server，找到 `tools/call` 的原始 JSON-RPC 请求和响应，对照第 1 节的时序图。
5. （进阶）加一个 prompt：`@mcp.prompt()` 装饰一个返回字符串的函数，在 Claude Code 里用 `/` 找到它。